In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from app.course import Course
from app.database import CourseDatabase

# Enable interactive plots
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load a Course from Database

In [ ]:
# Load available courses
db = CourseDatabase('../data/course_data.db')
available_courses = db.list_courses()
print("Available courses:")
for c in available_courses:
    print(f"  ID {c['id']}: {c['name']} ({c['course_type']})")

In [ ]:
# Choose a course to work with
COURSE_ID = 2  # Mt. Evans Hill Climb (more interesting for segmentation)

course_info = db.get_course(COURSE_ID)
course_df = db.get_course_data(COURSE_ID)

print(f"Loaded: {course_info['name']}")
print(f"Points: {len(course_df)}")
print(f"Total distance: {course_df['distance'].sum()/1000:.2f} km")
print(f"Elevation gain: {course_df[course_df['altitude_diff'] > 0]['altitude_diff'].sum():.0f} m")

In [ ]:
# Create Course object (skip_processing since data is already processed)
course = Course(course_df, skip_processing=True)
print(f"Course created with {len(course.processed_course)} points")

## 2. Visualize Raw Course Profile

In [ ]:
def plot_course_profile(df, title="Course Profile"):
    """Plot elevation and gradient profile."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Calculate cumulative distance in km
    cum_dist = df['distance'].cumsum() / 1000
    
    # Elevation profile
    ax1.fill_between(cum_dist, df['altitude'], alpha=0.3)
    ax1.plot(cum_dist, df['altitude'], 'b-', linewidth=0.5)
    ax1.set_ylabel('Altitude (m)')
    ax1.set_title(title)
    
    # Gradient profile
    gradient = df['gradient'] * 100  # Convert to percentage
    colors = ['green' if g >= 0 else 'red' for g in gradient]
    ax2.bar(cum_dist, gradient, width=0.05, color=colors, alpha=0.7)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Gradient (%)')
    ax2.set_xlabel('Distance (km)')
    ax2.set_ylim(-15, 15)
    
    plt.tight_layout()
    return fig

plot_course_profile(course.processed_course, f"{course_info['name']} - Raw Profile")
plt.show()

## 3. Auto Segmentation (Agglomerative Clustering)

Key parameters:
- **max_clusters**: Maximum number of segments (default: 10)
- **gradient_weight**: Weight for gradient in clustering (default: 1.0)
- **bearing_weight**: Weight for bearing changes (default: 0.5)
- **smoothing_window**: Rolling window for smoothing features (default: 5)

In [ ]:
# Try different auto-segmentation parameters
MAX_CLUSTERS = 10
GRADIENT_WEIGHT = 1.0
BEARING_WEIGHT = 0.5
SMOOTHING_WINDOW = 5

# Run auto-segmentation
course.auto_segmentation(
    max_clusters=MAX_CLUSTERS,
    gradient_weight=GRADIENT_WEIGHT,
    bearing_weight=BEARING_WEIGHT,
    smoothing_window=SMOOTHING_WINDOW
)

print(f"Created {len(course.course_segments)} segments")
print("\nSegment summary:")
for i, seg in enumerate(course.course_segments):
    dist_km = seg['distance'].sum() / 1000
    avg_grad = seg['gradient'].mean() * 100
    print(f"  Segment {i+1}: {dist_km:.2f} km, {avg_grad:+.1f}% avg gradient")

In [ ]:
def plot_segmented_profile(df, segments, title="Segmented Profile"):
    """Plot elevation profile with segment boundaries."""
    fig, ax = plt.subplots(figsize=(14, 6))
    
    cum_dist = df['distance'].cumsum() / 1000
    
    # Color each segment differently
    colors = plt.cm.tab10(np.linspace(0, 1, len(segments)))
    
    segment_start = 0
    for i, seg in enumerate(segments):
        seg_end = segment_start + len(seg)
        seg_cum_dist = cum_dist.iloc[segment_start:seg_end]
        seg_alt = df['altitude'].iloc[segment_start:seg_end]
        
        ax.fill_between(seg_cum_dist, seg_alt, alpha=0.4, color=colors[i], 
                       label=f'Seg {i+1}: {seg["gradient"].mean()*100:+.1f}%')
        ax.plot(seg_cum_dist, seg_alt, color=colors[i], linewidth=1)
        
        # Add vertical line at segment boundary
        if i > 0:
            ax.axvline(x=seg_cum_dist.iloc[0], color='black', linestyle='--', alpha=0.3)
        
        segment_start = seg_end
    
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Altitude (m)')
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=8)
    
    plt.tight_layout()
    return fig

plot_segmented_profile(course.processed_course, course.course_segments,
                       f"Auto Segmentation (max_clusters={MAX_CLUSTERS})")
plt.show()

## 4. Parameter Sensitivity Analysis

Let's see how different parameter values affect segmentation.

In [ ]:
# Test different max_clusters values
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

cluster_values = [5, 8, 12, 15]

for ax, max_clust in zip(axes, cluster_values):
    # Reset and re-segment
    test_course = Course(course_df, skip_processing=True)
    test_course.auto_segmentation(max_clusters=max_clust)
    
    cum_dist = test_course.processed_course['distance'].cumsum() / 1000
    colors = plt.cm.tab10(np.linspace(0, 1, len(test_course.course_segments)))
    
    segment_start = 0
    for i, seg in enumerate(test_course.course_segments):
        seg_end = segment_start + len(seg)
        seg_cum_dist = cum_dist.iloc[segment_start:seg_end]
        seg_alt = test_course.processed_course['altitude'].iloc[segment_start:seg_end]
        
        ax.fill_between(seg_cum_dist, seg_alt, alpha=0.4, color=colors[i])
        if i > 0:
            ax.axvline(x=seg_cum_dist.iloc[0], color='black', linestyle='--', alpha=0.3)
        segment_start = seg_end
    
    ax.set_title(f'max_clusters={max_clust} → {len(test_course.course_segments)} segments')
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Altitude (m)')

plt.suptitle('Effect of max_clusters on Segmentation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Test gradient vs bearing weight balance
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

weight_configs = [
    (1.0, 0.0, "Gradient only"),
    (1.0, 0.5, "Gradient + Light bearing"),
    (1.0, 1.0, "Equal weights"),
    (0.5, 1.0, "Bearing dominant")
]

for ax, (g_weight, b_weight, label) in zip(axes.flatten(), weight_configs):
    test_course = Course(course_df, skip_processing=True)
    test_course.auto_segmentation(
        max_clusters=10,
        gradient_weight=g_weight,
        bearing_weight=b_weight
    )
    
    cum_dist = test_course.processed_course['distance'].cumsum() / 1000
    colors = plt.cm.tab10(np.linspace(0, 1, len(test_course.course_segments)))
    
    segment_start = 0
    for i, seg in enumerate(test_course.course_segments):
        seg_end = segment_start + len(seg)
        seg_cum_dist = cum_dist.iloc[segment_start:seg_end]
        seg_alt = test_course.processed_course['altitude'].iloc[segment_start:seg_end]
        
        ax.fill_between(seg_cum_dist, seg_alt, alpha=0.4, color=colors[i])
        if i > 0:
            ax.axvline(x=seg_cum_dist.iloc[0], color='black', linestyle='--', alpha=0.3)
        segment_start = seg_end
    
    ax.set_title(f'{label}: grad={g_weight}, bear={b_weight}')
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Altitude (m)')

plt.suptitle('Effect of Gradient vs Bearing Weight', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Equal Segmentation

Simple alternative: divide course into N equal-distance segments.

In [ ]:
def equal_segmentation(course_df, n_segments):
    """Divide course into N equal-distance segments."""
    total_points = len(course_df)
    points_per_segment = total_points // n_segments
    
    segments = []
    for i in range(n_segments):
        start_idx = i * points_per_segment
        if i == n_segments - 1:
            end_idx = total_points  # Last segment gets remaining points
        else:
            end_idx = (i + 1) * points_per_segment
        
        segments.append(course_df.iloc[start_idx:end_idx].copy())
    
    return segments

# Compare auto vs equal segmentation
n_segs = 10
equal_segs = equal_segmentation(course.processed_course, n_segs)

print(f"Equal segmentation into {n_segs} segments:")
for i, seg in enumerate(equal_segs):
    dist_km = seg['distance'].sum() / 1000
    avg_grad = seg['gradient'].mean() * 100
    print(f"  Segment {i+1}: {dist_km:.2f} km, {avg_grad:+.1f}% avg gradient")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

cum_dist = course.processed_course['distance'].cumsum() / 1000

# Auto segmentation
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(course.course_segments)))
segment_start = 0
for i, seg in enumerate(course.course_segments):
    seg_end = segment_start + len(seg)
    seg_cum_dist = cum_dist.iloc[segment_start:seg_end]
    seg_alt = course.processed_course['altitude'].iloc[segment_start:seg_end]
    ax.fill_between(seg_cum_dist, seg_alt, alpha=0.4, color=colors[i])
    if i > 0:
        ax.axvline(x=seg_cum_dist.iloc[0], color='black', linestyle='--', alpha=0.5)
    segment_start = seg_end
ax.set_title(f'Auto Segmentation ({len(course.course_segments)} segments)')
ax.set_ylabel('Altitude (m)')

# Equal segmentation
ax = axes[1]
colors = plt.cm.tab10(np.linspace(0, 1, len(equal_segs)))
segment_start = 0
for i, seg in enumerate(equal_segs):
    seg_end = segment_start + len(seg)
    seg_cum_dist = cum_dist.iloc[segment_start:seg_end]
    seg_alt = course.processed_course['altitude'].iloc[segment_start:seg_end]
    ax.fill_between(seg_cum_dist, seg_alt, alpha=0.4, color=colors[i])
    if i > 0:
        ax.axvline(x=seg_cum_dist.iloc[0], color='black', linestyle='--', alpha=0.5)
    segment_start = seg_end
ax.set_title(f'Equal Segmentation ({len(equal_segs)} segments)')
ax.set_ylabel('Altitude (m)')
ax.set_xlabel('Distance (km)')

plt.suptitle('Auto vs Equal Segmentation Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Aggregate Segments

After segmentation, we aggregate to get summary statistics for optimization.

In [ ]:
# Use auto-segmented course
course.auto_segmentation(max_clusters=10)

# Aggregate segments
aggregated = course.aggregate_segments()

print("Aggregated segment data:")
display_cols = ['segment', 'distance_sum', 'gradient_mean', 'altitude_diff_sum', 'point_count']
available_cols = [c for c in display_cols if c in aggregated.columns]
aggregated[available_cols]

In [ ]:
# Visualize aggregated data
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distance per segment
ax = axes[0]
ax.bar(aggregated['segment'], aggregated['distance_sum']/1000, color='steelblue')
ax.set_xlabel('Segment')
ax.set_ylabel('Distance (km)')
ax.set_title('Segment Distances')

# Gradient per segment
ax = axes[1]
colors = ['green' if g >= 0 else 'red' for g in aggregated['gradient_mean']]
ax.bar(aggregated['segment'], aggregated['gradient_mean']*100, color=colors)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Segment')
ax.set_ylabel('Avg Gradient (%)')
ax.set_title('Segment Gradients')

# Elevation change per segment
ax = axes[2]
if 'altitude_diff_sum' in aggregated.columns:
    colors = ['green' if e >= 0 else 'red' for e in aggregated['altitude_diff_sum']]
    ax.bar(aggregated['segment'], aggregated['altitude_diff_sum'], color=colors)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Segment')
ax.set_ylabel('Elevation Change (m)')
ax.set_title('Segment Elevation Changes')

plt.suptitle('Aggregated Segment Summary', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

- **max_clusters** controls the maximum granularity - actual segments may be fewer
- **gradient_weight** emphasizes elevation changes (good for hilly courses)
- **bearing_weight** emphasizes direction changes (good for winding courses)
- Auto-segmentation creates segments with consistent terrain characteristics
- Equal segmentation is simpler but may split climbs/descents awkwardly
- Aggregated data is what gets passed to the optimizer